# Partitioning uncertainty by year instead of GWL

In [1]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import copy
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from matplotlib import colors as mcolors

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc
from funcs_processing import calc_uncerts
from funcs_plot import plot_map, vals_to_color_triangle

dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35503 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/35503/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/35503/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42943,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/35503/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33593,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/37741/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:38467,


2026-04-30 15:00:41,295 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 0d822da68de230cb63b78e3b616bc2b6 initialized by task ('rechunk-merge-rechunk-transfer-be16fd8df0c49f3b32a3aa2d5c1b46ef', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14) executed on worker tcp://127.0.0.1:34267
2026-04-30 15:00:48,296 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 0d822da68de230cb63b78e3b616bc2b6 deactivated due to stimulus 'task-finished-1777582848.2958539'
2026-04-30 15:02:19,712 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f3c5cdd6abfa387b91bfc0bca5ab8ef1 initialized by task ('rechunk-merge-rechunk-transfer-e47ac7d2f74c1cae23ca683ac5feae36', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0) executed on worker tcp://127.0.0.1:34267
2026-04-30 15:02:19,819 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f3c5cdd6abfa387b91bfc0bca5ab8ef1 deactivated due to stimulus 'task-finished-1777582939.8184657'
2026-04-30 15:02:27,454 - distributed.shuffle._sche

In [3]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':False},
               'gdppc':{'varname':'tassumpoly','geo_typ':'adm1','popwght':True},
              'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

#params_proc = {'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}


In [4]:
overwrite = True
ssp = 'ssp245'
for proc in params_proc:
    output_fn = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}_byyr.zarr'
    if overwrite or (not os.path.exists(output_fn)):
        #---------------------------
        # Setup
        #---------------------------
        # Load impacts projections
        fns = np.sort(glob.glob(f'{dir_list['proc']}impact_projs/{proc}/d{proc}*_agg-1deg_*ALLTIMEFRAMES*.zarr'))
        def load_impact(fn):
            ds_tmp = xr.open_zarr(fn,consolidated=False)
            ds_tmp = ds_tmp.set_coords('locid') # Should be coordinate, instead of concatenated variable
            ds_tmp = ds_tmp.assign_coords({'model':(('idv'),[re.split(r'\_',re.split(r'\/',fn)[-1])[2]]*ds_tmp.sizes['idv'])})
        
            ds_tmp['exprun'] = ds_tmp['experiment']+ds_tmp['run']
            ds_tmp = ds_tmp.reset_index('idv')
            ds_tmp = ds_tmp.set_index(idv = ['model','exprun'])
            return ds_tmp
        
        with warnings.catch_warnings():
            # Silence zarr warnings related to the /.done flags used
            warnings.filterwarnings('ignore')
            dss = xr.concat([load_impact(fn) for fn in fns],
                        dim='idv',join='outer')

        # SUBSET TO DESIRED SSP 
        dss = dss.isel(idv = (dss.experiment.load() == ssp))
        
        dss = dss.chunk({'poly_idx':50,'idv':-1,'proj_base':-1,'gwl':-1})

        # Save temporary rechunked file to make dask graphing more stable
        tmp_fn = dir_list['proc']+'tmp/uncert_rechunk_cache_tmp.zarr'
        dss = dss.reset_index('idv')
        for v in dss.coords:
            dss[v] = dss[v].load()
        utility_save(dss.drop_encoding(),tmp_fn,zarr_mode = 'w')
        dss = xr.open_zarr(tmp_fn)
        for v in ['model','exprun']:
            # Explicitly cast these dims as objects because of a numpy bug
            dss[v] = dss[v].astype(object)
        dss = dss.set_index(idv = ['model','exprun'])
        
        # Figure out how many BCD-ME members reach a given GWL
        # (copy.deepcopy() necessary as a workaround around a numpy bug
        # affecting the unstack a few lines down that's triggered 
        # indirectly by the reset_index here)
        gwl_counter = copy.deepcopy(dss.maxgwl).load().reset_index('idv').groupby(['model','maxgwl']).count()
        gwl_counter = gwl_counter.sel(maxgwl = gwl_counter.maxgwl.values[::-1]).cumsum('maxgwl')
        #print(gwl_counter)
        
        uncerts_all = []
        #---------------------------
        # Uncertainty partitioning over all runs
        #---------------------------
        dss_tmp = dss.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
        
        # Calculate uncertainty partitioning
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

        uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
        uncerts = uncerts.expand_dims({'sample':['allruns']})
        
        uncerts_all.append(uncerts)


        #---------------------------
        # Uncertainty partitioning over only models with > 1 run
        #---------------------------

        nruns = 3
        gwlmods = gwl_counter.sel(maxgwl=2070)
        proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
        
        dss_tmp = dss.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.sel(model=proc_mods)
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
        
        # Calculate uncertainty partitioning
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

        uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
        uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods']})
        
        uncerts_all.append(uncerts)

        #---------------------------
        # Uncertainty partitioning over only models with > 1 run,
        # but allowing different models over different timeperiods
        # if they fit the run limit
        #---------------------------
        nruns = 3
        dss_tmp = dss.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
        
        dss_tmp = dss_tmp.where((~np.isnan(dss_tmp)).sum('exprun')>=nruns)
        
        # Calculate uncertainty partitioning
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

        uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
        uncerts = uncerts.expand_dims({'sample':[f'atleast{str(nruns)}runs']})
        
        uncerts_all.append(uncerts)
        
        #---------------------------
        # Export
        #---------------------------
        # Concatenate into single ds
        uncerts_all = xr.concat(uncerts_all,dim='sample')
        uncerts_all = uncerts_all.set_coords('numruns')
        uncerts_all = uncerts_all.where(uncerts_all.sum('typ')>0)
        
        # Export
        uncerts_all.attrs = {'SOURCE':'4d_partition_uncertainty_byyr.ipynb',
                             'DESCRIPTION':f'Uncertainty partitioning of projections of {proc} between model, internal, and reanalysis sources'}
        
        utility_save(uncerts_all,output_fn,zarr_mode = 'w')

        os.system('rm -rf '+tmp_fn)
        print('Temporary file '+tmp_fn+' removed.')
    else:
        print(output_fn+' exists!')
    


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=13, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning i

/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of fre

/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_mortality_byyr.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


...

/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of fre

/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_payroll_byyr.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_payroll_byyr.zarr saved!
Temporary file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr removed.


In [ ]:
## Note: TaiESM1 is the odd one out on payroll that's causing model unc to blow up with the all runs 